In [ ]:
from pathlib import Path
from collections import Counter

DATA_ROOT = '/workspace/CSM/nonverbal_language/data/affectnet/archive (1)/YOLO_format'
CLASS_NAMES = ['Anger', 'Contempt', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

for split in ['train', 'valid']:
    lbl_dir = Path(DATA_ROOT) / split / 'labels'
    counter = Counter()
    for lbl in lbl_dir.glob('*.txt'):
        with open(lbl) as f:
            line = f.readline().strip()
            if line:
                counter[int(line.split()[0])] += 1
    print(f"\n[{split}]")
    for cls_id, cnt in sorted(counter.items()):
        print(f"  {CLASS_NAMES[cls_id]}: {cnt}개")


[train]
  Anger: 1123개
  Contempt: 729개
  Disgust: 718개
  Fear: 454개
  Happy: 1879개
  Neutral: 1107개
  Sad: 1001개
  Surprise: 1166개

[valid]
  Anger: 712개
  Contempt: 618개
  Disgust: 672개
  Fear: 622개
  Happy: 791개
  Neutral: 514개
  Sad: 603개
  Surprise: 874개


In [3]:
import cv2
import numpy as np
from pathlib import Path
from collections import Counter
import shutil
import random

DATA_ROOT = '/workspace/CSM/nonverbal_language/data/affectnet/archive (1)/YOLO_format'
OUTPUT_ROOT = '/workspace/CSM/nonverbal_language/data/affectnet/balanced'
CLASS_NAMES = ['Anger', 'Contempt', 'Disgust', 'Fear',
               'Happy', 'Neutral', 'Sad', 'Surprise']

random.seed(42)

def augment_image(img):
    """복제 시 적용할 증강 (랜덤 선택)"""
    aug_type = random.randint(0, 4)
    if aug_type == 0:
        # 좌우 반전
        return cv2.flip(img, 1)
    elif aug_type == 1:
        # 밝기 조절
        factor = random.uniform(0.7, 1.3)
        return np.clip(img * factor, 0, 255).astype(np.uint8)
    elif aug_type == 2:
        # 회전
        angle = random.uniform(-15, 15)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        return cv2.warpAffine(img, M, (w, h))
    elif aug_type == 3:
        # 좌우 반전 + 밝기
        img = cv2.flip(img, 1)
        factor = random.uniform(0.8, 1.2)
        return np.clip(img * factor, 0, 255).astype(np.uint8)
    else:
        # 노이즈 추가
        noise = np.random.randint(-20, 20, img.shape).astype(np.int16)
        return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

def balance_dataset(data_root, output_root, splits=['train', 'valid']):
    for split in splits:
        img_dir = Path(data_root) / split / 'images'
        lbl_dir = Path(data_root) / split / 'labels'
        
        out_img_dir = Path(output_root) / split / 'images'
        out_lbl_dir = Path(output_root) / split / 'labels'
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lbl_dir.mkdir(parents=True, exist_ok=True)
        
        # 클래스별 샘플 수집
        class_samples = {i: [] for i in range(8)}
        for img_path in sorted(img_dir.glob('*.jpg')):
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            with open(lbl_path) as f:
                line = f.readline().strip()
                if not line:
                    continue
                cls_id = int(line.split()[0])
            class_samples[cls_id].append((img_path, lbl_path))
        
        # 현재 분포 출력
        print(f"\n[{split}] 현재 분포:")
        for cls_id, samples in class_samples.items():
            print(f"  {CLASS_NAMES[cls_id]}: {len(samples)}개")
        
        # 최대 클래스 수로 맞춤
        target_count = max(len(v) for v in class_samples.values())
        print(f"\n목표 클래스당 샘플 수: {target_count}개")
        
        total_saved = 0
        for cls_id, samples in class_samples.items():
            # 원본 복사
            for img_path, lbl_path in samples:
                shutil.copy(img_path, out_img_dir / img_path.name)
                shutil.copy(lbl_path, out_lbl_dir / lbl_path.name)
            
            # 부족한 만큼 증강해서 채우기
            need = target_count - len(samples)
            for i in range(need):
                src_img, src_lbl = random.choice(samples)
                img = cv2.imread(str(src_img))
                aug_img = augment_image(img)
                
                # 새 파일명
                new_name = f"aug_{cls_id}_{i:05d}"
                cv2.imwrite(str(out_img_dir / f"{new_name}.jpg"), aug_img)
                shutil.copy(src_lbl, out_lbl_dir / f"{new_name}.txt")
            
            total_saved += target_count
            print(f"  {CLASS_NAMES[cls_id]}: {len(samples)} → {target_count}개 "
                  f"(+{need}개 증강)")
        
        print(f"[{split}] 총 {total_saved}개 저장 완료")

balance_dataset(DATA_ROOT, OUTPUT_ROOT)


[train] 현재 분포:
  Anger: 810개
  Contempt: 186개
  Disgust: 341개
  Fear: 355개
  Happy: 52개
  Neutral: 234개
  Sad: 669개
  Surprise: 595개

목표 클래스당 샘플 수: 810개
  Anger: 810 → 810개 (+0개 증강)
  Contempt: 186 → 810개 (+624개 증강)
  Disgust: 341 → 810개 (+469개 증강)
  Fear: 355 → 810개 (+455개 증강)
  Happy: 52 → 810개 (+758개 증강)
  Neutral: 234 → 810개 (+576개 증강)
  Sad: 669 → 810개 (+141개 증강)
  Surprise: 595 → 810개 (+215개 증강)
[train] 총 6480개 저장 완료

[valid] 현재 분포:
  Anger: 200개
  Contempt: 53개
  Disgust: 97개
  Fear: 73개
  Happy: 16개
  Neutral: 57개
  Sad: 177개
  Surprise: 134개

목표 클래스당 샘플 수: 200개
  Anger: 200 → 200개 (+0개 증강)
  Contempt: 53 → 200개 (+147개 증강)
  Disgust: 97 → 200개 (+103개 증강)
  Fear: 73 → 200개 (+127개 증강)
  Happy: 16 → 200개 (+184개 증강)
  Neutral: 57 → 200개 (+143개 증강)
  Sad: 177 → 200개 (+23개 증강)
  Surprise: 134 → 200개 (+66개 증강)
[valid] 총 1600개 저장 완료


## EfficientNet-B2 FINE-TUNING

In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
from pathlib import Path
from collections import Counter

CLASS_NAMES = ['Anger', 'Contempt', 'Disgust', 'Fear',
               'Happy', 'Neutral', 'Sad', 'Surprise']

class AffectNetYOLO(Dataset):
    def __init__(self, root, split, transform=None):
        self.transform = transform
        self.samples = []
        
        img_dir = Path(root) / split / 'images'
        lbl_dir = Path(root) / split / 'labels'
        
        for img_path in sorted(img_dir.glob('*.jpg')):
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            with open(lbl_path) as f:
                line = f.readline().strip()
                if not line:
                    continue
                cls_id = int(line.split()[0])
            self.samples.append((img_path, cls_id))
        
        print(f"[{split}] {len(self.samples)}개 샘플 로드됨")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, cls_id = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, cls_id

# ── 설정 ──────────────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '/workspace/CSM/nonverbal_language/data/affectnet/balanced'
NUM_CLASSES = 8
EPOCHS      = 50        # 늘림
BATCH_SIZE  = 32        # 줄임 (작은 데이터엔 작은 배치가 유리)
LR          = 1e-4      # 줄임 (오버피팅 방지)
IMG_SIZE    = 224       # 96→224 올림 (pretrained 모델 최적 해상도)

print(f"Device: {DEVICE}")

# ── 개선 1: 강한 증강으로 오버피팅 방지 ────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, 
                           saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3),  # 얼굴 일부 가리기
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_ds = AffectNetYOLO(DATA_ROOT, 'train', train_tf)
val_ds   = AffectNetYOLO(DATA_ROOT, 'valid', val_tf)

# ── 개선 2: WeightedRandomSampler로 클래스 불균형 보정 ──
label_counts = Counter([s[1] for s in train_ds.samples])
class_weights = {cls: 1.0/cnt for cls, cnt in label_counts.items()}
sample_weights = [class_weights[s[1]] for s in train_ds.samples]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, 
                          sampler=sampler,   # shuffle 대신 sampler
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

# ── 개선 3: 더 강한 모델 + Dropout ────────────────
model = models.efficientnet_b2(weights='IMAGENET1K_V1')  # b0→b2
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),          # 드롭아웃 추가
    nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
)
model = model.to(DEVICE)

# ── 개선 4: Label Smoothing으로 과신뢰 방지 ────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ── 학습 루프 ──────────────────────────────────────
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.detach().argmax(1) == labels).sum().item()
    
    # Validation
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            val_correct += (model(imgs).argmax(1) == labels).sum().item()
    
    train_acc = correct / len(train_ds)
    val_acc   = val_correct / len(val_ds)
    scheduler.step()
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Train: {train_acc:.3f} | Val: {val_acc:.3f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'class_names': CLASS_NAMES,
        }, 'best_emotion_model_v2.pth')
        print(f"  ✓ 저장 완료 (val_acc: {val_acc:.3f})")

print(f"\n최고 Val Acc: {best_val_acc:.3f}")

Device: cuda
[train] 6480개 샘플 로드됨
[valid] 1600개 샘플 로드됨
Epoch 01/50 | Loss: 1.9040 | Train: 0.292 | Val: 0.384
  ✓ 저장 완료 (val_acc: 0.384)
Epoch 02/50 | Loss: 1.4893 | Train: 0.515 | Val: 0.469
  ✓ 저장 완료 (val_acc: 0.469)
Epoch 03/50 | Loss: 1.2539 | Train: 0.635 | Val: 0.523
  ✓ 저장 완료 (val_acc: 0.523)
Epoch 04/50 | Loss: 1.1117 | Train: 0.706 | Val: 0.518
Epoch 05/50 | Loss: 0.9989 | Train: 0.765 | Val: 0.536
  ✓ 저장 완료 (val_acc: 0.536)
Epoch 06/50 | Loss: 0.9255 | Train: 0.804 | Val: 0.533
Epoch 07/50 | Loss: 0.8623 | Train: 0.832 | Val: 0.553
  ✓ 저장 완료 (val_acc: 0.553)
Epoch 08/50 | Loss: 0.8137 | Train: 0.855 | Val: 0.532
Epoch 09/50 | Loss: 0.7899 | Train: 0.868 | Val: 0.541
Epoch 10/50 | Loss: 0.7552 | Train: 0.890 | Val: 0.535
Epoch 11/50 | Loss: 0.7452 | Train: 0.890 | Val: 0.536
Epoch 12/50 | Loss: 0.7093 | Train: 0.906 | Val: 0.529
Epoch 13/50 | Loss: 0.6900 | Train: 0.914 | Val: 0.534
Epoch 14/50 | Loss: 0.6967 | Train: 0.910 | Val: 0.537
Epoch 15/50 | Loss: 0.6771 | Train: 0.92

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
from pathlib import Path
from collections import Counter

CLASS_NAMES = ['Anger', 'Contempt', 'Disgust', 'Fear',
               'Happy', 'Neutral', 'Sad', 'Surprise']

# ── DAN 모델 구현 ──────────────────────────────────
class AttentionHead(nn.Module):
    """단일 어텐션 헤드"""
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels, 1)
        self.bn   = nn.BatchNorm2d(in_channels)
    
    def forward(self, x):
        attn = torch.sigmoid(self.bn(self.conv(x)))
        return x * attn  # 어텐션 가중치 적용

class DAN(nn.Module):
    """
    Distract Your Attention Network
    - ResNet-18 backbone
    - 다중 어텐션 헤드로 다양한 표정 특징 포착
    """
    def __init__(self, num_classes=8, num_heads=4):
        super().__init__()
        resnet = models.resnet18(weights='IMAGENET1K_V1')
        
        # backbone: ResNet-18에서 마지막 fc 제거
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        # 출력: (B, 512, 7, 7)
        
        # 다중 어텐션 헤드
        self.heads = nn.ModuleList([
            AttentionHead(512) for _ in range(num_heads)
        ])
        
        # 헤드 가중치 (어떤 헤드가 더 중요한지 학습)
        self.head_weight = nn.Linear(num_heads, num_heads)
        
        self.gap = nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * num_heads, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
        self.num_heads = num_heads
    
    def forward(self, x):
        feat = self.backbone(x)  # (B, 512, 7, 7)
        
        head_outs = [self.gap(head(feat)) for head in self.heads]
        # 각 head_out: (B, 512, 1, 1)

        # 헤드 가중치 계산: 각 헤드의 평균값으로 중요도 점수 산출
        scores = torch.stack([
            h.view(h.size(0), -1).mean(dim=1)
            for h in head_outs
        ], dim=1)
        weights = torch.softmax(self.head_weight(scores), dim=1)  # (B, num_heads)sssssssssssssssssssssssssssssssssssssss

        # 가중 합산
        weighted = sum(w.unsqueeze(-1).unsqueeze(-1) * h
                    for w, h in zip(weights.unbind(dim=1), head_outs))
        # weighted: (B, 512, 1, 1)

        # concat에 weighted 정보 더하기
        concat = torch.cat(head_outs, dim=1)          # (B, 512*num_heads, 1, 1)
   
        final = torch.cat([concat, weighted], dim=1)

        out = self.classifier(final)
        return out

# ── 데이터셋 ───────────────────────────────────────
class AffectNetYOLO(Dataset):
    def __init__(self, root, split, transform=None):
        self.transform = transform
        self.samples = []
        img_dir = Path(root) / split / 'images'
        lbl_dir = Path(root) / split / 'labels'
        for img_path in sorted(img_dir.glob('*.jpg')):
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            with open(lbl_path) as f:
                line = f.readline().strip()
                if not line:
                    continue
                cls_id = int(line.split()[0])
            self.samples.append((img_path, cls_id))
        print(f"[{split}] {len(self.samples)}개 샘플 로드됨")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, cls_id = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, cls_id

# ── 설정 ──────────────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT   = '/workspace/CSM/nonverbal_language/data/affectnet/archive (1)/YOLO_format'
NUM_CLASSES = 8
BATCH_SIZE  = 32
IMG_SIZE    = 224

# 단계별 설정
STAGE1_EPOCHS = 5    # backbone freeze
STAGE2_EPOCHS = 15   # 마지막 블록 unfreeze
STAGE3_EPOCHS = 30   # 전체 unfreeze

print(f"Device: {DEVICE}")

# ── Transform ──────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── 데이터로더 ─────────────────────────────────────
train_ds = AffectNetYOLO(DATA_ROOT, 'train', train_tf)
val_ds   = AffectNetYOLO(DATA_ROOT, 'valid', val_tf)

label_counts   = Counter([s[1] for s in train_ds.samples])
class_weights  = {cls: 1.0/cnt for cls, cnt in label_counts.items()}
sample_weights = [class_weights[s[1]] for s in train_ds.samples]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, pin_memory=True)

# ── 모델 초기화 ────────────────────────────────────
model     = DAN(num_classes=NUM_CLASSES, num_heads=4).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ── 학습 함수 ──────────────────────────────────────
def train_stage(epochs, lr, stage_name):
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-3
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.01)
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (out.detach().argmax(1) == labels).sum().item()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                val_correct += (model(imgs).argmax(1) == labels).sum().item()

        train_acc = correct / len(train_ds)
        val_acc   = val_correct / len(val_ds)
        scheduler.step()

        print(f"[{stage_name}] Epoch {epoch+1:02d}/{epochs} | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Train: {train_acc:.3f} | Val: {val_acc:.3f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'class_names': CLASS_NAMES,
            }, 'best_emotion_model_DAN.pth')
            print(f"  ✓ 저장 완료 (val_acc: {val_acc:.3f})")
    
    return best_val_acc

# ════════════════════════════════════════════════
# 1단계: backbone freeze → attention head + classifier만 학습
# ════════════════════════════════════════════════
print("\n" + "="*50)
print("1단계: Backbone freeze")
print("="*50)

for param in model.backbone.parameters():
    param.requires_grad = False

train_stage(STAGE1_EPOCHS, lr=1e-3, stage_name="Stage1")


print("\n" + "="*50)
print("2단계: 마지막 블록 unfreeze")
print("="*50)

# ResNet-18 layer4만 unfreeze
for param in model.backbone[-1].parameters():
    param.requires_grad = True

train_stage(STAGE2_EPOCHS, lr=1e-4, stage_name="Stage2")


print("\n" + "="*50)
print("3단계: 전체 unfreeze")
print("="*50)

for param in model.parameters():
    param.requires_grad = True

best = train_stage(STAGE3_EPOCHS, lr=1e-5, stage_name="Stage3")
print(f"\n최종 최고 Val Acc: {best:.3f}")

Device: cuda
[train] 3242개 샘플 로드됨
[valid] 807개 샘플 로드됨
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%



1단계: Backbone freeze
[Stage1] Epoch 01/5 | Loss: 2.0699 | Train: 0.181 | Val: 0.208
  ✓ 저장 완료 (val_acc: 0.208)
[Stage1] Epoch 02/5 | Loss: 1.9638 | Train: 0.249 | Val: 0.221
  ✓ 저장 완료 (val_acc: 0.221)
[Stage1] Epoch 03/5 | Loss: 1.9117 | Train: 0.280 | Val: 0.196
[Stage1] Epoch 04/5 | Loss: 1.8630 | Train: 0.302 | Val: 0.196
[Stage1] Epoch 05/5 | Loss: 1.8590 | Train: 0.308 | Val: 0.219

2단계: 마지막 블록 unfreeze
[Stage2] Epoch 01/15 | Loss: 1.7750 | Train: 0.349 | Val: 0.287
  ✓ 저장 완료 (val_acc: 0.287)
[Stage2] Epoch 02/15 | Loss: 1.6720 | Train: 0.407 | Val: 0.353
  ✓ 저장 완료 (val_acc: 0.353)
[Stage2] Epoch 03/15 | Loss: 1.6083 | Train: 0.443 | Val: 0.364
  ✓ 저장 완료 (val_acc: 0.364)
[Stage2] Epoch 04/15 | Loss: 1.5629 | Train: 0.469 | Val: 0.371
  ✓ 저장 완료 (val_acc: 0.371)
[Stage2] Epoch 05/15 | Loss: 1.4926 | Train: 0.497 | Val: 0.394
  ✓ 저장 완료 (val_acc: 0.394)
[Stage2] Epoch 06/15 | Loss: 1.4632 | Train: 0.530 | Val: 0.418
  ✓ 저장 완료 (val_acc: 0.418)
[Stage2] Epoch 07/15 | Loss: 1.4094 | Tra